# Exercise 3: Protein Function Prediction with LSTM

**Lecture 10: Neural Networks and Deep Learning**

## Learning Objectives
1. Process variable-length protein sequences
2. Build bidirectional LSTM for sequence classification
3. Handle sequence padding and packing
4. Understand protein embedding

## Biological Context
Predict protein function family from amino acid sequence

In [ ]:
!pip install -q torch numpy matplotlib scikit-learn

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device}')

## Part 1: Generate Synthetic Protein Sequences

In [ ]:
# Amino acid vocabulary
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
aa_to_idx = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
idx_to_aa = {i: aa for aa, i in aa_to_idx.items()}

def generate_protein_sequence(min_len=50, max_len=200, function_class=0):
    '''Generate synthetic protein with class-specific patterns'''
    length = np.random.randint(min_len, max_len)
    # Add class-specific motifs
    if function_class == 0:  # Kinase
        motif = 'VAIK'
    elif function_class == 1:  # Protease
        motif = 'HDS'
    else:  # Transporter
        motif = 'GYG'
    
    seq = ''.join(np.random.choice(list(AMINO_ACIDS)) for _ in range(length))
    # Insert motif
    pos = np.random.randint(10, length-10)
    seq = seq[:pos] + motif + seq[pos+len(motif):]
    return seq

# Generate dataset
print('Generating 1000 protein sequences...')
sequences = []
labels = []
function_names = ['Kinase', 'Protease', 'Transporter']

for cls in range(3):
    for _ in range(333):
        seq = generate_protein_sequence(function_class=cls)
        sequences.append(seq)
        labels.append(cls)

print(f'Generated {len(sequences)} sequences')
print(f'Length range: {min(len(s) for s in sequences)}-{max(len(s) for s in sequences)}')
print(f'\nExample sequences:')
for i in range(3):
    print(f'{function_names[labels[i]]}: {sequences[i][:50]}...')

## Part 2: Preprocessing - Encoding and Padding

In [ ]:
def encode_sequence(seq):
    '''Convert amino acid sequence to indices'''
    return [aa_to_idx[aa] for aa in seq]

def pad_sequences(sequences, max_len=None):
    '''Pad sequences to same length'''
    if max_len is None:
        max_len = max(len(s) for s in sequences)
    
    padded = []
    lengths = []
    for seq in sequences:
        encoded = encode_sequence(seq)
        lengths.append(len(encoded))
        # Pad with zeros
        padded_seq = encoded + [0] * (max_len - len(encoded))
        padded.append(padded_seq)
    
    return np.array(padded), np.array(lengths)

# Encode sequences
X, lengths = pad_sequences(sequences)
y = np.array(labels)

print(f'Encoded shape: {X.shape}')
print(f'Lengths shape: {lengths.shape}')

# Split data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test, len_train, len_test = train_test_split(
    X, y, lengths, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTrain: {len(X_train)}, Test: {len(X_test)}')

## Part 3: Build Bidirectional LSTM

In [ ]:
class ProteinLSTM(nn.Module):
    '''Bidirectional LSTM for protein function prediction'''
    
    def __init__(self, vocab_size=20, embedding_dim=64, 
                 hidden_dim=128, num_layers=2, num_classes=3):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embedding_dim, hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            dropout=0.3,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim * 2, num_classes)  # *2 for bidirectional
    
    def forward(self, x, lengths):
        # x: [batch, seq_len]
        embedded = self.embedding(x)  # [batch, seq_len, embedding_dim]
        
        # Pack padded sequences for efficiency
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        
        # LSTM
        packed_output, (hidden, cell) = self.lstm(packed)
        
        # Use final hidden states from both directions
        # hidden: [num_layers*2, batch, hidden_dim]
        forward_hidden = hidden[-2, :, :]  # Last layer, forward
        backward_hidden = hidden[-1, :, :]  # Last layer, backward
        final_hidden = torch.cat([forward_hidden, backward_hidden], dim=1)
        
        return self.fc(final_hidden)

model = ProteinLSTM(vocab_size=20, embedding_dim=64, hidden_dim=128, num_classes=3)
model = model.to(device)

print(model)
print(f'\nParameters: {sum(p.numel() for p in model.parameters()):,}')

## Part 4: Training

In [ ]:
# Convert to tensors
X_train_t = torch.LongTensor(X_train)
y_train_t = torch.LongTensor(y_train)
len_train_t = torch.LongTensor(len_train)

X_test_t = torch.LongTensor(X_test)
y_test_t = torch.LongTensor(y_test)
len_test_t = torch.LongTensor(len_test)

train_dataset = torch.utils.data.TensorDataset(X_train_t, y_train_t, len_train_t)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print('Training...')
for epoch in range(30):
    model.train()
    total_loss = 0
    for batch_x, batch_y, batch_len in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        batch_len = batch_len.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x, batch_len)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}')

print('Training complete!')

## Part 5: Evaluation

In [ ]:
# Evaluate
model.eval()
with torch.no_grad():
    X_test_dev = X_test_t.to(device)
    len_test_dev = len_test_t.to(device)
    outputs = model(X_test_dev, len_test_dev)
    _, predicted = outputs.max(1)
    predictions = predicted.cpu().numpy()

from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns

acc = accuracy_score(y_test, predictions)
print(f'Test Accuracy: {acc*100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, predictions, target_names=function_names))

# Confusion matrix
cm = confusion_matrix(y_test, predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=function_names, yticklabels=function_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

print('\nKey Takeaways:')
print('1. Bidirectional LSTM captures context from both directions')
print('2. Sequence packing handles variable lengths efficiently')
print('3. Embedding layer learns meaningful amino acid representations')
print('4. Applications: function prediction, localization, structure')